# Notebook 12 — Differential Expression: DESeq2 Statistics

**Module:** 09 — Genomics and NGS  
**Notebook:** 12 of 16  
**Time estimate:** 90 minutes

> DESeq2 is the standard for RNA-seq differential expression. Implement the
> three core steps from scratch: size factor normalization, dispersion estimation,
> and the Wald test with BH correction.

---
## Step 1 — Motivation

We have raw count matrices from RNA-seq experiments and want to know: which genes
change significantly between two conditions? The t-test fails because RNA-seq counts
are not normally distributed (they are discrete, overdispersed, and heteroscedastic).
DESeq2 uses a negative binomial model that explicitly models biological variability
across samples.

---
## Step 2 — Intuition

**Why not a t-test?**  
RNA-seq counts for a gene across 3 replicates might be [1000, 980, 1100] for
control and [5, 8, 6] for treated. The t-test would find this significant, but
with only 3 replicates per group, we don't know if the variance is typical for
this gene. DESeq2 borrows variance information from all genes with similar
expression levels.

**DESeq2 three steps:**
1. **Size factor normalization:** Account for different sequencing depths across
   samples using the median-of-ratios method
2. **Dispersion estimation:** Estimate per-gene overdispersion, then shrink toward
   a smooth fit across all genes (empirical Bayes)
3. **Wald test:** Compare normalized counts between conditions; compute Wald statistic
   and p-value; apply BH correction for multiple testing

---
## Step 3 — Biological Background

**The negative binomial distribution for RNA-seq counts:**  
Why not Poisson? Poisson assumes variance = mean. In reality, RNA-seq counts are
overdispersed: variance > mean because of biological variability between replicates
in addition to sampling variability. The negative binomial adds a dispersion
parameter $\alpha$ (gene-specific):
$$\text{Var}(K_{gj}) = \mu_{gj} + \alpha_g \mu_{gj}^2$$

where $\mu_{gj}$ is the expected count for gene $g$ in sample $j$.

**Why median-of-ratios, not total count normalization?**  
Simple normalization (divide by total counts) is sensitive to a few highly
expressed genes. If one gene is massively upregulated in treated vs. control,
it inflates the total count, making all other genes appear downregulated.
Median-of-ratios is robust to outliers.

---
## Step 4 — Mathematical Explanation

**Step 1 — Size factor normalization (median-of-ratios):**

For $G$ genes and $J$ samples:
1. Compute log geometric mean of each gene across samples:
   $$\bar{m}_g = \frac{1}{J} \sum_j \log(k_{gj}) \quad (\text{geometric mean})$$
2. For each sample $j$, compute the log ratio relative to geometric mean:
   $$\log s_j = \text{median}_{g} \left( \log k_{gj} - \bar{m}_g \right)$$
   (exclude genes with zero counts in any sample)
3. Size factor: $s_j = \exp(\log s_j)$
4. Normalized count: $\tilde{k}_{gj} = k_{gj} / s_j$

**Step 3 — Wald test:**

For each gene, fit a generalized linear model (GLM):
$$\log \mu_{gj} = \beta_{g0} + \beta_{g1} x_j + \log s_j$$

where $x_j = 1$ if sample is treated, $0$ if control, and $s_j$ is the size factor
(offset). The Wald statistic:
$$W_g = \frac{\hat{\beta}_{g1}}{SE(\hat{\beta}_{g1})}$$

Under the null, $W_g \sim N(0, 1)$. P-value = $2 \cdot \Phi(-|W_g|)$.

**Benjamini-Hochberg correction:**
Sort p-values: $p_{(1)} \leq p_{(2)} \leq \ldots \leq p_{(m)}$. Adjusted p-value:
$$p^{\text{adj}}_{(i)} = \min_{j \geq i} \left( \frac{m}{j} p_{(j)} \right)$$
Reject all hypotheses where $p^{\text{adj}}_{(i)} \leq \alpha$.

In [ ]:
# Step 6 — Python: DESeq2 core steps from scratch
import numpy as np
from scipy import stats
from scipy.optimize import minimize_scalar


def deseq2_size_factors(count_matrix: np.ndarray) -> np.ndarray:
    """
    DESeq2 median-of-ratios size factor estimation.
    count_matrix: shape (n_genes, n_samples)
    Returns: size factors, shape (n_samples,)
    """
    # Exclude genes with zero in any sample
    mask = (count_matrix > 0).all(axis=1)
    filtered = count_matrix[mask]
    if filtered.shape[0] == 0:
        raise ValueError("No genes with non-zero counts in all samples")

    # Log geometric mean of each gene across samples
    log_geo_means = np.log(filtered).mean(axis=1)  # shape (n_filtered_genes,)

    # For each sample, compute median log ratio
    log_ratios = np.log(filtered) - log_geo_means[:, np.newaxis]  # (genes, samples)
    log_size_factors = np.median(log_ratios, axis=0)  # (samples,)

    return np.exp(log_size_factors)


def p_adjust_bh(pvalues: np.ndarray) -> np.ndarray:
    """
    Benjamini-Hochberg FDR correction.
    Returns adjusted p-values.
    """
    m = len(pvalues)
    order = np.argsort(pvalues)
    sorted_p = pvalues[order]

    # Compute adjusted values: min(m/i * p_i) from right to left
    adjusted = np.minimum(1.0, sorted_p * m / (np.arange(1, m+1)))
    # Enforce monotonicity: take cumulative min from right
    for i in range(m - 2, -1, -1):
        adjusted[i] = min(adjusted[i], adjusted[i+1])

    # Return in original order
    result = np.empty(m)
    result[order] = adjusted
    return result


def deseq2_wald_test(
    count_matrix: np.ndarray,
    condition: np.ndarray,
    size_factors: np.ndarray = None,
    min_count: int = 5
) -> dict:
    """
    Simplified DESeq2 Wald test for DE analysis.
    count_matrix: (n_genes, n_samples)
    condition: 0 = control, 1 = treatment
    Returns: {log2fc, pvalue, padj, mean_count, gene_idx}
    """
    n_genes, n_samples = count_matrix.shape

    if size_factors is None:
        size_factors = deseq2_size_factors(count_matrix)

    # Normalize counts
    norm_counts = count_matrix / size_factors[np.newaxis, :]

    # Filter low-count genes
    mean_counts = norm_counts.mean(axis=1)
    keep = mean_counts >= min_count

    cond0 = (condition == 0)
    cond1 = (condition == 1)

    log2fc = np.full(n_genes, np.nan)
    pvalue = np.full(n_genes, np.nan)

    for g in np.where(keep)[0]:
        y0 = norm_counts[g, cond0]
        y1 = norm_counts[g, cond1]

        # Log2 fold change
        mean0 = y0.mean() + 0.5  # pseudocount
        mean1 = y1.mean() + 0.5
        log2fc[g] = np.log2(mean1 / mean0)

        # Wald test on log-normalized counts (approximation)
        # Real DESeq2 uses GLM; this is a t-test on log-counts as teaching approximation
        log_y0 = np.log2(y0 + 0.5)
        log_y1 = np.log2(y1 + 0.5)
        if log_y0.std() == 0 and log_y1.std() == 0:
            pvalue[g] = 1.0
        else:
            _, p = stats.ttest_ind(log_y0, log_y1, equal_var=False)
            pvalue[g] = p

    # BH correction on non-NaN values
    padj = np.full(n_genes, np.nan)
    valid = ~np.isnan(pvalue)
    if valid.sum() > 0:
        padj[valid] = p_adjust_bh(pvalue[valid])

    return {
        'log2fc': log2fc,
        'pvalue': pvalue,
        'padj': padj,
        'mean_count': mean_counts,
        'size_factors': size_factors,
    }


# Simulate count matrix: 1000 genes, 6 samples (3 control, 3 treated)
rng = np.random.default_rng(42)
n_genes = 1000
n_ctrl, n_treat = 3, 3
condition = np.array([0]*n_ctrl + [1]*n_treat)

# Base counts
base_expr = rng.exponential(200, n_genes).clip(10, 5000)
size_factors_true = np.array([1.0, 1.2, 0.9, 1.1, 0.95, 1.05])

# Plant 100 DE genes: 60 up, 40 down
de_genes = rng.choice(n_genes, 100, replace=False)
true_lfc = np.zeros(n_genes)
true_lfc[de_genes[:60]] = rng.normal(2, 0.5, 60)
true_lfc[de_genes[60:]] = rng.normal(-2, 0.5, 40)

counts = np.zeros((n_genes, n_ctrl + n_treat), dtype=int)
for j in range(n_ctrl + n_treat):
    cond = condition[j]
    fold_change = 2 ** (true_lfc if cond == 1 else np.zeros(n_genes))
    expected = base_expr * fold_change * size_factors_true[j]
    counts[:, j] = rng.negative_binomial(expected, 0.1).clip(0, None)  # dispersion~0.1

# Run DESeq2
sf = deseq2_size_factors(counts)
results = deseq2_wald_test(counts, condition, sf)

# Summary
sig = ~np.isnan(results['padj']) & (results['padj'] < 0.05) & (np.abs(results['log2fc']) > 1)
n_sig = sig.sum()
tp = np.intersect1d(np.where(sig)[0], de_genes)
fp = np.setdiff1d(np.where(sig)[0], de_genes)
fn = np.setdiff1d(de_genes, np.where(sig)[0])

print(f"Estimated size factors: {sf.round(3)}")
print(f"True size factors:      {size_factors_true}")
print()
print(f"Significant DE genes (padj<0.05, |log2FC|>1): {n_sig}")
print(f"True positives:  {len(tp)} / {len(de_genes)} planted DE genes ({100*len(tp)/len(de_genes):.1f}% sensitivity)")
print(f"False positives: {len(fp)} / {n_genes - len(de_genes)} null genes")
print(f"False negatives: {len(fn)} planted DE genes missed")

In [ ]:
# Step 7 — Visualization: MA plot, volcano plot, normalized count distributions
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

lfc = results['log2fc']
mean_c = results['mean_count']
padj = results['padj']
valid = ~np.isnan(lfc) & ~np.isnan(padj)

# Panel A: MA plot
ax = axes[0, 0]
sig_mask = valid & (padj < 0.05) & (np.abs(lfc) > 1)
ax.scatter(np.log2(mean_c[valid & ~sig_mask] + 1), lfc[valid & ~sig_mask],
           alpha=0.3, s=5, color='gray', label='Not significant')
ax.scatter(np.log2(mean_c[sig_mask] + 1), lfc[sig_mask],
           alpha=0.8, s=10, color='tomato', label=f'DE (n={sig_mask.sum()})')
ax.axhline(0, color='black', lw=0.5)
ax.axhline(1, color='blue', lw=0.5, ls='--', alpha=0.5)
ax.axhline(-1, color='blue', lw=0.5, ls='--', alpha=0.5)
ax.set_xlabel('Mean expression (log2)')
ax.set_ylabel('log2 fold change')
ax.set_title('A. MA plot: log2FC vs. mean expression')
ax.legend(fontsize=8)

# Panel B: Volcano plot
ax2 = axes[0, 1]
neg_log_p = -np.log10(padj[valid] + 1e-300)
up = valid & (padj < 0.05) & (lfc > 1)
dn = valid & (padj < 0.05) & (lfc < -1)
ns = valid & ~(padj < 0.05)
ax2.scatter(lfc[ns], neg_log_p[valid & ns[valid]], alpha=0.3, s=5, color='gray')
ax2.scatter(lfc[up], -np.log10(padj[up] + 1e-300), alpha=0.8, s=10, color='tomato',
            label=f'Up ({up.sum()})')
ax2.scatter(lfc[dn], -np.log10(padj[dn] + 1e-300), alpha=0.8, s=10, color='steelblue',
            label=f'Down ({dn.sum()})')
ax2.axhline(-np.log10(0.05), color='orange', ls='--', lw=1, label='padj=0.05')
ax2.axvline(1, color='gray', ls=':', lw=0.8)
ax2.axvline(-1, color='gray', ls=':', lw=0.8)
ax2.set_xlabel('log2 fold change')
ax2.set_ylabel('-log10(adjusted p-value)')
ax2.set_title('B. Volcano plot')
ax2.legend(fontsize=8)

# Panel C: Before/after normalization
ax3 = axes[1, 0]
norm_counts = counts / sf[np.newaxis, :]
log_raw = np.log1p(counts)
log_norm = np.log1p(norm_counts)
colors_c = ['#4FC3F7','#4FC3F7','#4FC3F7','#FF8A65','#FF8A65','#FF8A65']
bp1 = ax3.boxplot([log_raw[:, j] for j in range(6)], positions=np.arange(6)*2.5,
                   patch_artist=True, widths=0.9,
                   boxprops={'facecolor': 'lightgray'}, medianprops={'color':'black'})
bp2 = ax3.boxplot([log_norm[:, j] for j in range(6)], positions=np.arange(6)*2.5+1,
                   patch_artist=True, widths=0.9,
                   boxprops={'facecolor': 'steelblue'}, medianprops={'color':'red'})
ax3.set_xticks(np.arange(6)*2.5 + 0.5)
ax3.set_xticklabels([f'S{j+1}' for j in range(6)], fontsize=9)
ax3.set_ylabel('log(count + 1)')
ax3.set_title('C. Count distributions before (gray) and\nafter normalization (blue)')
ax3.legend([bp1['boxes'][0], bp2['boxes'][0]], ['Raw', 'Normalized'], fontsize=8)

# Panel D: BH correction effect
ax4 = axes[1, 1]
pvals = results['pvalue'][valid]
padjs = results['padj'][valid]
ax4.scatter(pvals, padjs, alpha=0.3, s=5, color='steelblue')
ax4.plot([0, 1], [0, 1], 'k--', lw=1, label='y=x (no correction)')
ax4.axhline(0.05, color='orange', ls='--', lw=1, label='FDR=0.05')
ax4.set_xlabel('Raw p-value')
ax4.set_ylabel('BH-adjusted p-value')
ax4.set_title('D. BH correction: raw vs. adjusted p-values\n(adjusted always ≥ raw)')
ax4.legend(fontsize=8)

plt.suptitle('DESeq2 Differential Expression Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('deseq2_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Three samples have raw total counts [12M, 18M, 10M]. Using DESeq2's size factor
   approach (not total count), compute size factors for gene A with counts [100, 150, 83].
2. Verify: after applying size factors, what are the normalized counts?
3. Implement `p_adjust_bh` again from scratch without looking at the implementation
   above. Test it on p-values [0.001, 0.01, 0.05, 0.2, 0.5] with 5 hypotheses.
4. Why does DESeq2 use a negative binomial model instead of a Poisson model?
   Describe a biological scenario where the difference would matter.

---
## Step 10 — Quiz

1. Describe the DESeq2 size factor method in 3 sentences.
2. Why does DESeq2 use raw counts rather than TPM as input?
3. What is the Wald test statistic and what distribution does it follow under the null?
4. What does FDR=5% mean after BH correction?
5. A gene has log2FC=3 but padj=0.8. What does this mean biologically?

---
## Step 12 — Reflection

> *[In one paragraph: explain what 'shrinkage' of log2FC estimates does in DESeq2
> (lfcShrink) and why it is especially important for low-count genes. This was not
> implemented above — look it up and explain it in your own words.]*

---
*Next: `13_genome_browsers_igv_and_variant_visualization.ipynb`*